# Mixture of Agents (MoA)

> Notebook generated from [`examples/patterns/05_mixture_of_agents.py`](../../examples/patterns/05_mixture_of_agents.py).

| Field | Value |
|------|-------|
| **Dataset** | MedQA USMLE (embedded, 4 questions) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** Per question: answers from the N proposers, aggregator synthesis, metrics (`providers_used`, layers) and verification of the correct option.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Mixture of Agents (MoA) — Medical QA with multiple models
==========================================================
Pattern: SPEC-PAT-006 / prismal.agents.patterns.mixture_of_agents

Dataset: MedQA (USMLE Medical Board Questions)
  • 12,723 USMLE-style multiple-choice questions.
  • Reference: https://huggingface.co/datasets/GBaker/MedQA-USMLE-4-options
  • Why: MoA shines when different models have different "blind spots".
    In medicine, the diversity of perspectives reduces clinical errors.
    The aggregator synthesizes the best of each proposer.

Pattern description:
  - Proposers: N models answer the question independently
    and in parallel (different providers/models bring different perspectives).
  - Aggregator: A higher-capacity model synthesizes the responses into
    a single cohesive answer and improves collective accuracy.
  - Fault tolerance: if some proposers fail, the rest continue;
    only fails if ALL proposers fail.
  - Multi-layer: n_aggregator_layers > 1 allows iterative refinements.

Usage:
    uv run python examples/patterns/05_mixture_of_agents.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio

from prismal.agents.patterns.mixture_of_agents import MixtureOfAgents, MoAResult

## Dataset: MedQA USMLE questions

In [ ]:
# Representative sample of USMLE Step 1/2 level questions.
MEDQA_SAMPLES = [
    {
        "id": "MQ1",
        "category": "Pharmacology",
        "question": (
            "A 65-year-old patient with paroxysmal atrial fibrillation is "
            "taking warfarin. They are prescribed amoxicillin for a urinary "
            "tract infection. What is the most likely effect on the INR?"
        ),
        "options": {
            "A": "INR will decrease significantly",
            "B": "INR will increase due to reduction of vitamin K-producing gut flora",
            "C": "INR will not change",
            "D": "INR will decrease due to enzymatic induction",
        },
        "correct": "B",
    },
    {
        "id": "MQ2",
        "category": "Pathophysiology",
        "question": (
            "A 45-year-old woman presents with fatigue, weight gain, cold "
            "intolerance and constipation. Lab work shows elevated TSH and low "
            "free T4. What is the most likely diagnosis?"
        ),
        "options": {
            "A": "Primary hyperthyroidism",
            "B": "Primary hypothyroidism",
            "C": "Secondary hypothyroidism",
            "D": "Cushing's syndrome",
        },
        "correct": "B",
    },
    {
        "id": "MQ3",
        "category": "Microbiology",
        "question": (
            "A 5-year-old child presents with fever, a pruritic vesicular "
            "rash beginning on the trunk and spreading to the periphery, and "
            "a positive Koebner sign. What is the most likely etiologic agent?"
        ),
        "options": {
            "A": "Measles virus (Morbillivirus)",
            "B": "Varicella-zoster virus (VZV)",
            "C": "Parvovirus B19",
            "D": "Streptococcus pyogenes",
        },
        "correct": "B",
    },
]


def format_question(sample: dict) -> str:
    """Format a MedQA question for the LLM."""
    options_text = "\n".join(f"  {k}) {v}" for k, v in sample["options"].items())
    return (
        f"Medical question ({sample['category']}):\n\n"
        f"{sample['question']}\n\n"
        f"Options:\n{options_text}\n\n"
        "Please reason through your answer step by step and choose the correct option."
    )


async def run_moa(sample: dict, models: list[str], aggregator: str) -> MoAResult:
    """Run Mixture of Agents on a MedQA question.

    Args:
        sample: Question with options and correct answer.
        models: List of proposer models.
        aggregator: Aggregator model.

    Returns:
        MoAResult with the final synthesized answer.
    """
    moa = MixtureOfAgents(
        proposer_models=models,
        aggregator_model=aggregator,
        n_aggregator_layers=1,  # one aggregation layer
    )

    return await moa.generate(
        query=format_question(sample),
        state={"messages": [], "metadata": {"category": sample["category"]}},
    )


async def main() -> None:
    print("=" * 70)
    print("  Mixture of Agents — Dataset: MedQA USMLE (Medical QA)")
    print("=" * 70)

    # Model configuration (adjust to available providers)
    # MoA is more powerful with DIFFERENT models (different biases)
    proposer_models = [
        "claude-sonnet-4-6",  # Anthropic
        "claude-sonnet-4-6",  # In production: use gpt-4o, gemini-1.5-pro
        "claude-sonnet-4-6",  # In production: use specialized medical models
    ]
    aggregator_model = "claude-sonnet-4-6"  # In production: claude-opus-4-6

    print(f"\n  Proposers : {len(proposer_models)} parallel models")
    print(f"  Aggregator: {aggregator_model}")
    print("  MoA layers: 1")
    print()

    correct_count = 0

    for sample in MEDQA_SAMPLES:
        print(f"[{sample['id']}] Category: {sample['category']}")
        print(f"  Question: {sample['question'][:80]}...")
        print(f"  Correct answer: {sample['correct']}) {sample['options'][sample['correct']]}")

        result = await run_moa(sample, proposer_models, aggregator_model)

        print("\n  MoA answer:")
        print(f"  {result.final_answer[:300]}")

        print("\n  MoA metrics:")
        print(f"    Successful proposers : {len(result.layer_outputs[0])}/{len(proposer_models)}")
        print(f"    Layers completed     : {len(result.layer_outputs)}")
        print(f"    Providers used       : {result.providers_used}")

        # Check whether the correct answer appears in the final answer
        correct_letter = sample["correct"]
        if correct_letter in result.final_answer:
            print(f"    ✓ The correct option ({correct_letter}) is in the answer")
            correct_count += 1
        else:
            print(f"    ✗ The correct option ({correct_letter}) was not clearly detected")

        print()
        print("-" * 70)

    print(f"\n  MoA accuracy: {correct_count}/{len(MEDQA_SAMPLES)} questions")
    print(
        "\n  [Note] In production, using different proposer models "
        "maximizes diversity and collective accuracy (Wang et al. 2024)."
    )

    # Demonstrate proposer fault tolerance
    print("\n[Proposer fault tolerance]")
    print("  If 1 of 3 proposers fails → MoA continues with the remaining 2")
    print("  If 2 of 3 proposers fail  → MoA continues with the remaining 1")
    print("  If 3 of 3 proposers fail  → MoAError (total failure)")

    # Demonstrate multi-layer MoA
    print("\n[Multi-layer MoA (n_aggregator_layers=2)]")
    moa_multilayer = MixtureOfAgents(
        proposer_models=proposer_models[:2],
        aggregator_model=aggregator_model,
        n_aggregator_layers=2,
    )
    sample = MEDQA_SAMPLES[0]
    result_ml = await moa_multilayer.generate(
        query=format_question(sample),
        state={},
    )
    print(f"  Layer 1 proposers   : {len(result_ml.layer_outputs[0])} responses")
    if len(result_ml.layer_outputs) > 1:
        print(f"  Layer 2 refinement  : {len(result_ml.layer_outputs[1])} responses")
    print(f"  Final answer        : {result_ml.final_answer[:150]}...")


if __name__ == "__main__":
    asyncio.run(main())


---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()